<a href="https://colab.research.google.com/github/suri307/MLA0205-FOML/blob/main/EXP%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
from collections import Counter

data = [
    ["Sunny", "Hot", "High", "Weak", "No"],
    ["Sunny", "Hot", "High", "Strong", "No"],
    ["Overcast", "Hot", "High", "Weak", "Yes"],
    ["Rain", "Mild", "High", "Weak", "Yes"],
    ["Rain", "Cool", "Normal", "Weak", "Yes"],
    ["Rain", "Cool", "Normal", "Strong", "No"],
    ["Overcast", "Cool", "Normal", "Strong", "Yes"],
    ["Sunny", "Mild", "High", "Weak", "No"],
    ["Sunny", "Cool", "Normal", "Weak", "Yes"],
    ["Rain", "Mild", "Normal", "Weak", "Yes"],
    ["Sunny", "Mild", "Normal", "Strong", "Yes"],
    ["Overcast", "Mild", "High", "Strong", "Yes"],
    ["Overcast", "Hot", "Normal", "Weak", "Yes"],
    ["Rain", "Mild", "High", "Strong", "No"]
]

attributes = ["Outlook", "Temperature", "Humidity", "Wind"]

def entropy(rows):
    total = len(rows)
    counts = Counter(row[-1] for row in rows)

    result = 0
    for count in counts.values():
        probability = count / total
        result -= probability * math.log2(probability)

    return result

def information_gain(rows, index):
    total_entropy = entropy(rows)
    values = set(row[index] for row in rows)

    weighted_entropy = 0

    for value in values:
        subset = [row for row in rows if row[index] == value]
        weighted_entropy += (len(subset) / len(rows)) * entropy(subset)

    return total_entropy - weighted_entropy

def id3(rows, remaining_attributes):
    labels = [row[-1] for row in rows]

    if len(set(labels)) == 1:
        return labels[0]

    if not remaining_attributes:
        return Counter(labels).most_common(1)[0][0]

    best_attribute = max(
        remaining_attributes,
        key=lambda index: information_gain(rows, index)
    )

    tree = {
        "attribute": attributes[best_attribute],
        "branches": {}
    }

    for value in sorted(set(row[best_attribute] for row in rows)):
        subset = [row for row in rows if row[best_attribute] == value]

        new_attributes = [
            index for index in remaining_attributes
            if index != best_attribute
        ]

        tree["branches"][value] = id3(subset, new_attributes)

    return tree

def print_tree(tree, level=0):
    if isinstance(tree, str):
        print("  " * level + "-> " + tree)
        return

    print("  " * level + tree["attribute"])

    for value, branch in tree["branches"].items():
        print("  " * (level + 1) + value + ":")
        print_tree(branch, level + 2)

def predict(tree, sample):
    if isinstance(tree, str):
        return tree

    attribute_index = attributes.index(tree["attribute"])
    value = sample[attribute_index]

    if value not in tree["branches"]:
        return "Unknown"

    return predict(tree["branches"][value], sample)

decision_tree = id3(data, list(range(len(attributes))))

print("Decision Tree:")
print_tree(decision_tree)

test_sample = ["Sunny", "Cool", "High", "Strong"]

print("\nNew Sample:", test_sample)
print("Predicted Class:", predict(decision_tree, test_sample))

Decision Tree:
Outlook
  Overcast:
    -> Yes
  Rain:
    Wind
      Strong:
        -> No
      Weak:
        -> Yes
  Sunny:
    Humidity
      High:
        -> No
      Normal:
        -> Yes

New Sample: ['Sunny', 'Cool', 'High', 'Strong']
Predicted Class: No
